# E1.11 · Model risk management for AI systems

**Function E — AI for GRC → The GRC Practitioner (Risk & Control)**  ·  *Security of AI*

Builds on **[E1.10 · The stakeholder map: who owns what](https://spbreed.github.io/cyber-commons/lessons/E1.10.html)**.

| | |
|---|---|
| Open-source tooling | Inspect |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Model risk management has forty years of doctrine on validating models — conceptual soundness, ongoing monitoring, independent validation. Most of it transfers. The part that does not is the part where the model calls tools.

## 2 · The framework

```
   SR 11-7 lineage                 what an agent adds
   +----------------------+        +-----------------------+
   | conceptual soundness |  ok    | it calls tools        |
   | ongoing monitoring   |  ok    | it acts on the world  |
   | independent validation| ok    | output is not a number|
   +----------------------+        +-----------------------+

   most of forty years of doctrine transfers. the tool call does not.
```

Model risk management is not new. The SR 11-7 lineage has governed models in
regulated institutions for over a decade, and its three pillars are sound:

1. **Conceptual soundness** — is the method appropriate for the purpose?
2. **Ongoing monitoring** — is it still performing as validated?
3. **Independent validation** — did someone other than the builder check?

All three still hold for AI systems. What breaks is not the framework but a
silent assumption underneath it: **that a model produces an output, and a human
decides what to do with it.**

Once the model can call a tool, that assumption is void. Validation scoped to
the model's *predictions* says nothing about the model's *actions*. You can hold
a perfectly valid validation report for a system that has since been granted
write access to a production database, and nothing in the classical process is
required to notice.

So the extension is narrow and specific: the unit of validation becomes the
**model plus its tool surface plus its autonomy level**, and any change to any of
the three triggers revalidation — not just a change to the weights.

## 3 · The three pillars, and what each assumes

In [ ]:
PILLARS = {
 "conceptual_soundness": ("is the method appropriate for the purpose",
                          "assumes the purpose is stable and stated"),
 "ongoing_monitoring":   ("is it still performing as validated",
                          "assumes performance is what changes"),
 "independent_validation":("did someone other than the builder check",
                          "assumes the thing checked is the thing deployed"),
}
print(f"{'pillar':24s}{'what it asks':46s}what it quietly assumes")
for p in sorted(PILLARS):
    asks, assumes = PILLARS[p]
    print(f"{p:24s}{asks:46s}{assumes}")
print()
print("All three survive contact with AI. The assumptions are what break.")

## 4 · The unit of validation, before and after tools

In [ ]:
VALIDATED = {
 "model": "glm-5.2", "version": "2026-03",
 "purpose": "summarise support tickets",
 "tools": [],                       # at validation time it had none
 "autonomy": "L1",                  # suggests; a human acts
}

def validation_covers(deployed, validated):
    diffs = []
    if deployed["model"] != validated["model"]:       diffs.append("model changed")
    if deployed["version"] != validated["version"]:   diffs.append("version changed")
    if deployed["purpose"] != validated["purpose"]:   diffs.append("purpose changed")
    if sorted(deployed["tools"]) != sorted(validated["tools"]):
        diffs.append(f"tool surface changed: {sorted(set(deployed['tools']) - set(validated['tools']))}")
    if deployed["autonomy"] != validated["autonomy"]: diffs.append(
        f"autonomy raised {validated['autonomy']} -> {deployed['autonomy']}")
    return (not diffs), diffs

DEPLOYED = dict(VALIDATED, tools=["read_ticket", "write_ticket", "db_update"],
                autonomy="L3")
ok, diffs = validation_covers(DEPLOYED, VALIDATED)
print(f"validation still covers what is deployed: {ok}")
for d in diffs:
    print(f"   {d}")
print()
print("Same weights. Same version. The validation report is accurate about a")
print("system that no longer exists, and nothing in the classical process is")
print("required to notice, because the classical trigger is a model change.")
assert not ok

## 5 · Where it breaks — monitoring the wrong thing well

In [ ]:
import random
def monitor(metric, runs=200, seed=4):
    rng = random.Random(seed)
    return [round(rng.gauss(0.92, 0.01), 3) for _ in range(runs)]

acc = monitor("summarisation_accuracy")
print(f"summarisation accuracy over 200 runs: mean {sum(acc)/len(acc):.3f}, "
      f"min {min(acc)}, max {max(acc)}")
print("threshold 0.85 -> breaches:", sum(a < 0.85 for a in acc))
print()
UNMONITORED = ["rows written to production", "tools invoked per run",
               "actions taken without human review", "scope of the credential used"]
print("what is NOT on the dashboard:")
for u in UNMONITORED:
    print(f"   {u}")
print()
print("The monitoring is excellent and it is monitoring the prediction. The")
print("risk moved to the action, and the action has no threshold, no baseline")
print("and no alert.")
assert sum(a < 0.85 for a in acc) == 0

## 6 · The control — revalidate on the triple, not on the weights

In [ ]:
TRIGGERS = {
 "model or version change": True,
 "prompt or config change": True,
 "tool added or scope widened": True,
 "autonomy level raised": True,
 "purpose changed": True,
 "calendar year elapsed": True,
}
CLASSICAL = {"model or version change", "calendar year elapsed"}

print(f"{'trigger':32s}{'classical MRM':16s}extended")
for t in TRIGGERS:
    print(f"{t:32s}{'yes' if t in CLASSICAL else 'no':16s}yes")
missed = [t for t in TRIGGERS if t not in CLASSICAL]
print(f"\ntriggers classical MRM would miss: {len(missed)}")
for m in missed: print(f"   {m}")
print()
ok2, diffs2 = validation_covers(DEPLOYED, VALIDATED)
print(f"under the extended triggers, this deployment requires revalidation: {not ok2}")
print(f"reasons: {diffs2}")
assert len(missed) == 4

## 7 · Verify — what a validation record must now carry

In [ ]:
record = {
 "model": DEPLOYED["model"], "version": DEPLOYED["version"],
 "purpose": DEPLOYED["purpose"],
 "tool_surface": sorted(DEPLOYED["tools"]),
 "autonomy": DEPLOYED["autonomy"],
 "validated_unit": "model + tool surface + autonomy",
 "monitors": ["summarisation_accuracy", "rows_written", "tools_per_run",
              "actions_without_review"],
 "revalidation_triggers": sorted(TRIGGERS),
 "independent_of_builder": True,
}
for k in sorted(record):
    print(f"   {k:24s}{record[k]}")
print()
print("Three fields carry the whole extension: validated_unit, tool_surface and")
print("autonomy. Without them a validation report describes a text generator,")
print("and the thing in production is an actor.")
assert record["validated_unit"].startswith("model + tool")

## What you just proved

The three SR 11-7 pillars, each with the assumption it quietly makes. A system validated with no tools at L1 is shown deployed with three tools at L3 — same model, same version — and the validation no longer covers it. Monitoring reports 200 clean runs of summarisation accuracy while four action-level metrics have no threshold at all, and four revalidation triggers classical MRM would miss are named.

## Your turn

Take one validated model in your estate and list the tools it holds today. If any of them post-dates the validation report, the report is describing a different system.

---

**Next → [E1.12 · Working the seams](https://spbreed.github.io/cyber-commons/lessons/E1.12.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.11.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.11.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*